<a href="https://colab.research.google.com/github/cysorianoc/IBM_Machine_Learning/blob/main/Course_1_ML_NB_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Foundation

## Section 1, Part b: Reading Data Lab


In [ ]:
# Imports
import sqlite3 as sq3
import pandas.io.sql as pds
import pandas as pd

## Lab Exercise: Reading in database files

 - Create a variable, `path`, containing the path to the `baseball.db` contained in `resources/`
 - Create a connection, `con`, that is connected to database at `path`
 - Create a variable, `query`, containing a SQL query which reads in all data from the `allstarfull` table
 - Create a variable, `observations`, by using pandas' [read_sql](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_sql.html?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0232ENSkillsNetwork837-2023-01-01)

### Optional
 - Create a variable, `tables`, which reads in all data from the table `sqlite_master`
 - Pretend that you were interesting in creating a new baseball hall of fame. Join and analyze the tables to evaluate the top 3 all time best baseball players.


In [ ]:
# Download the database
!wget -P data https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML0232EN-SkillsNetwork/asset/baseball.db

--2025-07-07 14:16:27--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML0232EN-SkillsNetwork/asset/baseball.db
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 198.23.119.245
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|198.23.119.245|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7626752 (7.3M) [binary/octet-stream]
Saving to: ‘data/baseball.db’

baseball.db         100%[===================>]   7.27M  16.5MB/s    in 0.4s    

2025-07-07 14:16:28 (16.5 MB/s) - ‘data/baseball.db’ saved [7626752/7626752]



In [ ]:
### BEGIN SOLUTION
# Create a variable, `path`, containing the path to the `baseball.db` contained in `resources/`

path = '/content/data/baseball.db'

# Create a connection, `con`, that is connected to database at `path`
con = sq3.Connection(path)

# Create a variable, `query`, containing a SQL query which reads in all data from the `` table

query = """
SELECT *
    FROM allstarfull
    ;
"""

allstar_observations = pd.read_sql(query, con)

# print(allstar_observations)

# Create a variable, tables, which reads in all data from the table sqlite_master
all_tables = pd.read_sql('SELECT * FROM sqlite_master', con)
print(all_tables)

# Pretend that you were interesting in creating a new baseball hall of fame. Join and analyze the tables to evaluate the top 3 all time best baseball players
best_query = """
SELECT playerID, sum(GP) AS num_games_played, AVG(startingPos) AS avg_starting_position
    FROM allstarfull
    GROUP BY playerID
    ORDER BY num_games_played DESC, avg_starting_position ASC
    LIMIT 3
"""
best = pd.read_sql(best_query, con)
print(best.head())
### END SOLUTION

    type                  name     tbl_name  rootpage  \
0  table           allstarfull  allstarfull         2   
1  index  ix_allstarfull_index  allstarfull         3   
2  table               schools      schools        26   
3  index      ix_schools_index      schools        31   
4  table               batting      batting        99   
5  index      ix_batting_index      batting       100   

                                                 sql  
0  CREATE TABLE "allstarfull" (\n"index" INTEGER,...  
1  CREATE INDEX "ix_allstarfull_index"ON "allstar...  
2  CREATE TABLE "schools" (\n"index" INTEGER,\n  ...  
3  CREATE INDEX "ix_schools_index"ON "schools" ("...  
4  CREATE TABLE "batting" (\n"index" INTEGER,\n  ...  
5  CREATE INDEX "ix_batting_index"ON "batting" ("...  
    playerID  num_games_played  avg_starting_position
0  musiast01              24.0               6.357143
1   mayswi01              24.0               8.000000
2  aaronha01              24.0               8.470588

---
### Machine Learning Foundation (C) 2020 IBM Corporation


# Additional ideas

Extract all the tables inside the database

In [ ]:
# Step 1: List all tables in the database. In case of not knowing what is inside
tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables_df = pd.read_sql(tables_query, con)
print("Available tables in the database:")
print(tables_df)

Available tables in the database:
          name
0  allstarfull
1      schools
2      batting


In [ ]:
# Step 2: run a query from one of the tables and crete a dataframe
query = """
SELECT *
    FROM allstarfull
    ;
"""

allstar_observations = pd.read_sql(query, con)

allstar_observations.head()

,index,playerID,yearID,gameNum,gameID,teamID,lgID,GP,startingPos
0,0,gomezle01,1933,0,ALS193307060,NYA,AL,1.0,1.0
1,1,ferreri01,1933,0,ALS193307060,BOS,AL,1.0,2.0
2,2,gehrilo01,1933,0,ALS193307060,NYA,AL,1.0,3.0
3,3,gehrich01,1933,0,ALS193307060,DET,AL,1.0,4.0
4,4,dykesji01,1933,0,ALS193307060,CHA,AL,1.0,5.0


In [ ]:
# Step 3: Hall of Fame query

best_query = """
SELECT playerID, sum(GP) AS num_games_played, AVG(startingPos) AS avg_starting_position
    FROM allstarfull
    GROUP BY playerID
    ORDER BY num_games_played DESC, avg_starting_position ASC
    LIMIT 3
"""
best = pd.read_sql(best_query, con)
print(best.head())

    playerID  num_games_played  avg_starting_position
0  musiast01              24.0               6.357143
1   mayswi01              24.0               8.000000
2  aaronha01              24.0               8.470588


In [ ]:
# ALTERNATIVE USING THE PANDAS DATAFRAME
# Step 1: Group by 'playerID' and calculate the aggregations
hall_of_fame = (
    allstar_observations
    .groupby('playerID', as_index=False)
    .agg(
        num_games_played=('GP', 'sum'),
        avg_starting_position=('startingPos', 'mean')
    )
)

# Step 2: Sort by num_games_played (descending) and avg_starting_position (ascending)
hall_of_fame_sorted = hall_of_fame.sort_values(
    by=['num_games_played', 'avg_starting_position'],
    ascending=[False, True]
)

# Step 3: Select top 3 players
top_3 = hall_of_fame_sorted.head(3)

# Display result
print(top_3)


       playerID  num_games_played  avg_starting_position
1145  musiast01              24.0               6.357143
1030   mayswi01              24.0               8.000000
0     aaronha01              24.0               8.470588


# An interactive table viewer

To have a glance of what is inside the .db and then perform specific queries

In [ ]:
import sqlite3 as sq3
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Connect to the SQLite database
path = '/content/data/baseball.db'
con = sq3.connect(path)

# Get table names
tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables_df = pd.read_sql(tables_query, con)
table_names = tables_df['name'].tolist()

# Create dropdown widget
table_dropdown = widgets.Dropdown(
    options=table_names,
    description='Select Table:',
    disabled=False,
)

# Define what happens when a table is selected
def display_table(selected_table):
    query = f"SELECT * FROM {selected_table} LIMIT 10;"
    df = pd.read_sql(query, con)
    display(df)

# Create an interactive output
widgets.interact(display_table, selected_table=table_dropdown);


interactive(children=(Dropdown(description='Select Table:', options=('allstarfull', 'schools', 'batting'), val…

In [ ]:
# For example a query from schools table to get the top three states with the highest number of schools

query = """
SELECT state, COUNT(*) AS school_count
FROM schools  # choose the target table
GROUP BY state
ORDER BY school_count DESC
LIMIT 3;
"""

top_states_df = pd.read_sql(query, con)  #we can save it as a dataframe
display(top_states_df)


,state,school_count
0,CA,136
1,TX,80
2,PA,72


## Another example

Let`s try to get the data from a database found in a web repository

I just googled .db example:

[Source](https://www.timestored.com/data/sample/sqlite)

In [ ]:
# Step 1
# Download the database
!wget -P data2 https://www.timestored.com/data/sample/sakila.db  #Saved in data2 folder

--2025-07-07 14:35:33--  https://www.timestored.com/data/sample/sakila.db
Resolving www.timestored.com (www.timestored.com)... 139.162.217.116
Connecting to www.timestored.com (www.timestored.com)|139.162.217.116|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5828608 (5.6M)
Saving to: ‘data2/sakila.db’

sakila.db           100%[===================>]   5.56M  4.14MB/s    in 1.3s    

2025-07-07 14:35:35 (4.14 MB/s) - ‘data2/sakila.db’ saved [5828608/5828608]



In [ ]:
# Step 2: define the database path (where is located) and create a connection object
# Create a variable, `path`, containing the path to the `baseball.db` contained in `resources/`

path = '/content/data2/sakila.db'

# Create a connection, `con`, that is connected to database at `path`
con = sq3.Connection(path)

# Create a variable, `query`, containing a SQL query which reads in all data from the `` table

tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables_df = pd.read_sql(tables_query, con)
print("Available tables in the database:")
print(tables_df)

Available tables in the database:
             name
0           actor
1         country
2            city
3         address
4        language
5        category
6        customer
7            film
8      film_actor
9   film_category
10      film_text
11      inventory
12          staff
13          store
14        payment
15         rental


In [ ]:
# Step 3: I think we can put here the interactive table viewer

import sqlite3 as sq3
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Connect to the SQLite database
path = '/content/data2/sakila.db'
con = sq3.connect(path)

# Get table names
tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables_df = pd.read_sql(tables_query, con)
table_names = tables_df['name'].tolist()

# Create dropdown widget
table_dropdown = widgets.Dropdown(
    options=table_names,
    description='Select Table:',
    disabled=False,
)

# Define what happens when a table is selected
def display_table(selected_table):
    query = f"SELECT * FROM {selected_table} LIMIT 20;"
    df = pd.read_sql(query, con)
    display(df)

# Create an interactive output
widgets.interact(display_table, selected_table=table_dropdown);


interactive(children=(Dropdown(description='Select Table:', options=('actor', 'country', 'city', 'address', 'l…

In [ ]:
# Select  table and do a simple query
# For example a query from film_category table to get the 5 more persistent categories

query = """
SELECT category_id, COUNT(*) AS category_count
FROM film_category
GROUP BY category_id
ORDER BY category_count DESC
LIMIT 5;
"""

top_categories_df = pd.read_sql(query, con)
display(top_categories_df)

,category_id,category_count
0,15,74
1,9,73
2,8,69
3,6,68
4,2,66


from matplotlib import pyplot as plt
top_categories_df['category_id'].plot(kind='hist', bins=20, title='category_id')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
top_categories_df['category_count'].plot(kind='hist', bins=20, title='category_count')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
top_categories_df.plot(kind='scatter', x='category_id', y='category_count', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
top_categories_df['category_id'].plot(kind='line', figsize=(8, 4), title='category_id')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
top_categories_df['category_count'].plot(kind='line', figsize=(8, 4), title='category_count')
plt.gca().spines[['top', 'right']].set_visible(False)